In [1]:
import pandas as pd 
import numpy as np
import sqlite3
import requests 
import csv
import time
from SPARQLWrapper import SPARQLWrapper, JSON

In [4]:
# #opening the fantasy csv file 

fant = pd.read_csv(("fantasybooks.csv"))


#slicing the dataframe into the column i need

authors = fant["Author"]


#saving it to a new csv file 

authors.to_csv('authors_fantasy.csv', index = False, encoding = 'utf-8')

authors_fant = pd.read_csv('authors_fantasy.csv')
#authors_fant


#splitting the authors into single strings 

#authors_fant['Author'] = authors_fant['Author'].str.split(" /  ")#.str.join(", ").str.replace(" ' ", "")
authors_fant['Author'] = authors_fant['Author'].str.replace('/', ' / ')  # Ensure spaces around the slash
authors_fant['Author'] = authors_fant['Author'].str.replace(r'\s+', ' ', regex=True)  # Replace multiple spaces with a single space
authors_fant['Author'] = authors_fant['Author'].str.strip()
authors_fant['Author'] = authors_fant['Author'].str.split(" / ")


authors_fant

,Author
0,[J.K. Rowling / Mary GrandPré]
1,[J.K. Rowling / Mary GrandPré]
2,[J.K. Rowling]
3,[J.K. Rowling / Mary GrandPré]
4,[J.K. Rowling / Mary GrandPré]
...,...
2545,[C.S. Lewis / Pauline Baynes / Ana Falcão Bastos]
2546,[C.S. Lewis / Pauline Baynes / Ana Falcão Bastos]
2547,[C.S. Lewis / Pauline Baynes / Ana Falcão Bastos]
2548,[William T. Vollmann]


In [5]:
#creating a series 

authors_fantas = authors_fant["Author"]
print(authors_fantas)


0                          [J.K. Rowling / Mary GrandPré]
1                          [J.K. Rowling / Mary GrandPré]
2                                          [J.K. Rowling]
3                          [J.K. Rowling / Mary GrandPré]
4                          [J.K. Rowling / Mary GrandPré]
                              ...                        
2545    [C.S. Lewis / Pauline Baynes / Ana Falcão Bastos]
2546    [C.S. Lewis / Pauline Baynes / Ana Falcão Bastos]
2547    [C.S. Lewis / Pauline Baynes / Ana Falcão Bastos]
2548                                [William T. Vollmann]
2549                                [William T. Vollmann]
Name: Author, Length: 2550, dtype: object


In [10]:
#since my series contains single strings with many authors in it
#i am opening the dataframe series as a list, looping throuhg it
#splitting the strings through the / marker, so that in the list i have only single strings per author
#since in the dataframe its not possible to keep different values inside the same column that aren't inside a list
#i could split the lists of authors into multiple columns but that looks messy for now


#preparing an emoty set
#since sets don't have duplicates 
final_authors = set()


#nested loop since we have lists with multiple authors
# looping through each list of authors in authors_fantas
for author_list in authors_fantas:
    # looping through each individual author in the list
    for author_name in author_list:
       #looping through eachh string in the list and split authors by '/'
        individual_authors = [author.strip() for author in author_name.split(' / ')]
        # putting the authors into a set so i remove duplicayes
        final_authors.update(individual_authors)

# converting the set back to a list
final_authors = list(final_authors)

print(final_authors)


['Terence Tunberg', 'John Ridgway', 'Peter Vaughan', "Kevin O'Malley", 'Chris Van Allsburg', '橋口 たかし', 'Roy Hudd', 'Shana Abe', 'Kristine Kathryn Rusch', 'Lucius Shepard', 'Emile Martel', 'Frank E. Peretti', 'Joachim Körber', 'Carrie Channell', 'Peter David', 'Matthew Sturges', 'Emma Holly', 'George Alec Effinger', 'Laszlo Kubinyi', 'Leon Stover', 'Daniel Feldman', 'Adam Rex', 'Ned Dameron', 'Hans J. Schütz', 'Robert Stephens', 'Catherynne M. Valente', 'Mike Dringenberg', 'Horace Walpole', 'David Luke', 'K. Tempest Bradford', 'Robin Wasserman', 'Astrid Lindgren', 'Steve Yeowell', 'Adi Granov', 'John Cassaday', 'Lin Carter', 'Isak Dinesen', 'Scott Hampton', 'Daniela Gamba', 'Edwin A. Abbott', 'Robert Mayer', 'Brian Michael Bendis', 'K.A. Applegate', 'James Kahn', 'Brian Ajhar', 'David Bevington', 'Diane Dillon', 'Lan Medina', 'Guido Waldman', 'Lucy Corvino', 'Jim Woodring', 'E.R. Eddison', 'Pat Lee', 'Guy Parker-Rees', 'John Bunyan', 'Eric Flint', 'Carlos Castañeda', 'Haruki Murakami', 

In [11]:
#link for the query 

sparql = SPARQLWrapper("https://query.wikidata.org/sparql")


In [ ]:
#function to query through all the authors names in my list

def query_wikidata(name):
    url = 'https://query.wikidata.org/sparql'
    sparql_query = f"""
    prefix schema: <http://schema.org/>
    SELECT ?person ?personLabel ?dob ?nationalityLabel
    WHERE {{
        ?person wdt:P31 wd:Q5;
                rdfs:label "{name}"@en;
                wdt:P569 ?dob;
                wdt:P27 ?nationality.
        SERVICE wikibase:label {{ bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }}
    }}
    LIMIT 1
    """
    
    r = requests.get(url, params={'format': 'json', 'query': sparql_query})
    
    if r.status_code == 200:
        return r.json()
    else:
        print(f"Error for {name}: {r.status_code}")
        return None

# Loop through each row in your DataFrame (assuming 'Author(s)' is the column with lists of auth
for name in final_authors:  # Iterate over each author in the list
        
        # Execute query with retry logic
        result = None
        attempts = 0
        while result is None and attempts < 2:
            result = query_wikidata(name)
            
            if result is None:
                attempts += 1
                wait_time = 2 ** attempts  # Exponential backoff
                print(f"Retrying in {wait_time} seconds for {name}...")
                time.sleep(wait_time)
        
        if result:
            # Process the result (example: print or store)
            print(f"Result for {name}: {result}")
        
        time.sleep(2)

Result for Terence Tunberg: {'head': {'vars': ['person', 'personLabel', 'dob', 'nationalityLabel']}, 'results': {'bindings': [{'person': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q1215530'}, 'dob': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime', 'type': 'literal', 'value': '1950-08-12T00:00:00Z'}, 'personLabel': {'xml:lang': 'en', 'type': 'literal', 'value': 'Terence Tunberg'}, 'nationalityLabel': {'xml:lang': 'en', 'type': 'literal', 'value': 'United States of America'}}]}}
Result for John Ridgway: {'head': {'vars': ['person', 'personLabel', 'dob', 'nationalityLabel']}, 'results': {'bindings': [{'person': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q6254890'}, 'dob': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime', 'type': 'literal', 'value': '1940-05-04T00:00:00Z'}, 'personLabel': {'xml:lang': 'en', 'type': 'literal', 'value': 'John Ridgway'}, 'nationalityLabel': {'xml:lang': 'en', 'type': 'literal', 'value': 'United Kingdom'}}]}}
Result

In [ ]:
#next step would be to save this results into either a json file or a csv
#the csv then needs to be flattened --> since the query result is full of nested dictionaries
#TRY to merge the two csv files so that i don't lose the correspondence between authors
#the main csv file is still clean, the authors aren't split neither in lists nor in single strings --> could try to 
#split the authors into several columns, in that case the query can iterate through several columns 





In [ ]:
def query_wikidata(name):
    url = 'https://query.wikidata.org/sparql'
    sparql_query = f"""
    prefix schema: <http://schema.org/>
    SELECT ?person ?personLabel ?dob ?nationalityLabel
    WHERE {{
        ?person wdt:P31 wd:Q5;
                rdfs:label "{name}"@en;
                wdt:P569 ?dob;
                wdt:P27 ?nationality.
        SERVICE wikibase:label {{ bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }}
    }}
    LIMIT 1
    """
    
    r = requests.get(url, params={'format': 'json', 'query': sparql_query})
    
    if r.status_code == 200:
        return r.json()
    else:
        print(f"Error for {name}: {r.status_code}")
        return None


for authors_list in authors:  
    for name in authors_list:  # Iterate over each author in the list
        result = None
        attempts = 0
        while result is None and attempts < 2:
            result = query_wikidata(name)
            
            if result is None:
                attempts += 1
                wait_time = 2 ** attempts  # Exponential backoff
                print(f"Retrying in {wait_time} seconds for {name}...")
                time.sleep(wait_time)
        
        if result:
            # Process the result as needed
            print(f"Result for {name}: {result}")
        
        time.sleep(2) 

In [8]:
#since my series contains single strings with many authors in it
#i am opening the dataframe series as a list, looping throuhg it
#splitting the strings through the / marker, so that in the list i have only single strings per author
#since in the dataframe its not possible to keep different values inside the same column that aren't inside a list
#i could split the lists of authors into multiple columns but that looks messy for now

#preparing an emoty set
#since sets don't have duplicates 
final_authors = set() 

#looping through eachh string in the list and split authors by '/'
for author_name in authors_fant:  
    #removing the spaces
        authors = [author_name.strip() for authorname in authors_fant]#.split(' / ')]
    
final_authors.update(authors)

# converting the set back to a list
final_authors = list(final_authors)

#the list is ready to query 
print(final_authors)

['Author']


In [ ]:
# Loop through each name and add dynamic delay with retry logic
for name in authors:
    result = None
    attempts = 0
    while result is None and attempts < 2:  
        result = query_wikidata(name)
        
        if result is None:
            attempts += 1
            wait_time = 2 ** attempts  # Exponential backoff (wait longer after each failure)
            print(f"Retrying in {wait_time} seconds...")
            time.sleep(wait_time)
    
    if result:
        # Process the result as before
        print(result)
    time.sleep(2)

In [ ]:
for name in column_values:
    # Prepare the SPARQL query, formatting the name into the query string
    sparql_query = f"""
    prefix schema: <http://schema.org/>
    SELECT ?person ?personLabel ?dob ?nationalityLabel
    WHERE {{
        ?person wdt:P31 wd:Q5;          # Must be a human
                rdfs:label "{name}"@en; # Match the name
                wdt:P569 ?dob;          # Date of birth
                wdt:P27 ?nationality.   # Nationality (country of citizenship)
        SERVICE wikibase:label {{ bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }}
    }}
    LIMIT 1
    """


url = 'https://query.wikidata.org/sparql'

r = requests.get(url, params={'format': 'json', 'query': sparql_query})

if r.status_code == 200:
        data = r.json()  # Parse the JSON response
        
        # Check if any results were returned
        if 'results' in data and 'bindings' in data['results'] and len(data['results']['bindings']) > 0:
            # Extract the relevant information
            person_data = data['results']['bindings'][0]  # Get the first result
            person_name = person_data['personLabel']['value']
            dob = person_data['dob']['value']
            nationality = person_data['nationalityLabel']['value']
            
            # Print or store the results
            print(f"Name: {person_name}")
            print(f"DOB: {dob}")
            print(f"Nationality: {nationality}")
            print("-" * 30)
        else:
            print(f"No data found for {name}")
    else:
        print(f"Failed to retrieve data for {name}. Status code: {r.status_code}")


data = r.json()
print(data['results']['bindings'])

In [ ]:
import pandas as pd
import requests

# Step 1: Load your main CSV (books with authors in list format)
main_df = pd.read_csv('books.csv')

# Step 2: Load the author info CSV (Wikidata results)
author_info_df = pd.read_csv('author_info.csv')

# Step 3: Write a function to query Wikidata and return the information for a given author
def query_wikidata(name):
    url = 'https://query.wikidata.org/sparql'
    sparql_query = f"""
    prefix schema: <http://schema.org/>
    SELECT ?person ?personLabel ?dob ?nationalityLabel
    WHERE {{
        ?person wdt:P31 wd:Q5;
                rdfs:label "{name}"@en;
                wdt:P569 ?dob;
                wdt:P27 ?nationality.
        SERVICE wikibase:label {{ bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }}
    }}
    LIMIT 1
    """
    
    r = requests.get(url, params={'format': 'json', 'query': sparql_query})
    
    if r.status_code == 200:
        return r.json()
    else:
        print(f"Error for {name}: {r.status_code}")
        return None

# Step 4: Define a function to get author details from the author_info_df
def get_author_details(author_name):
    author_details = author_info_df[author_info_df['author_name'] == author_name]
    if not author_details.empty:
        return author_details.iloc[0]['dob'], author_details.iloc[0]['nationality']
    return None, None

# Step 5: Process each row in the main dataframe and query for author details
author_data = []
for idx, row in main_df.iterrows():
    authors = eval(row['authors'])  # Convert the string representation of the list back to a list
    book_title = row['book_title']
    
    for author in authors:
        # Query Wikidata for the author
        result = query_wikidata(author)
        
        if result:
            # Extract the necessary details from the result
            dob = result['results']['bindings'][0]['dob']['value'] if 'dob' in result['results']['bindings'][0] else None
            nationality = result['results']['bindings'][0]['nationalityLabel']['value'] if 'nationalityLabel' in result['results']['bindings'][0] else None
            author_data.append([book_title, author, dob, nationality])
        else:
            # If Wikidata query fails, just append the author name with None for details
            author_data.append([book_title, author, None, None])

# Step 6: Create a new DataFrame with the results
author_df = pd.DataFrame(author_data, columns=['book_title', 'author_name', 'dob', 'nationality'])

# Step 7: Merge the main DataFrame with the author data
final_df = pd.merge(main_df, author_df, how='left', on='book_title')

# Step 8: Save the result to a new CSV file
final_df.to_csv('merged_books_authors.csv', index=False)

print("Merged CSV created successfully!")
